In [46]:
import pandas as pd

In [47]:
df = pd.read_csv('dados/dados_fonte/municipios_bspn_base.csv', sep=';')
aplicabilidade_verificacoes = pd.read_csv('dados/dados_fonte/Aplicabilidade_verificacoes.csv', sep=';')
descricao_verificacoes = pd.read_csv('dados/dados_fonte/Descricao_verificacoes.csv', sep=';')


In [48]:
df_descricao = descricao_verificacoes[['no_verificacao', 'no_desc']]
df_descricao.rename(columns={'no_verificacao':'verificacao', 'no_desc':'descricao'}, inplace=True)
df_descricao.head(5)

,verificacao,descricao
0,D1_00001,Homologação de todos os RREOs
1,D1_00002,Homologação da DCA
2,D1_00003,Homologação de todos os RGFs do poder Executivo
3,D1_00004,Homologação de todos os RGFs do poder Legislativo
4,D1_00005,"Homologação de todos os RGFs do Judiciário, Mi..."


In [49]:
df_aplicabilidade = aplicabilidade_verificacoes.rename(columns={'VERIFICACAO':'verificacao'})
df_aplicabilidade.fillna('', inplace=True)
df_aplicabilidade.head()

,verificacao,2025,2024,2023,2022,2021,2020,2019
0,D1_00001,T,T,T,T,T,T,T
1,D1_00002,T,T,T,T,T,T,T
2,D1_00003,T,T,T,T,T,T,T
3,D1_00004,T,T,T,T,T,T,T
4,D1_00005,E,E,E,E,E,E,E


In [ ]:
import re

verificacoes = [
    coluna for coluna in df.columns
    if re.match(r'^D[1-4]_\d+$', coluna)
]

data_verificacoes = df.melt(
    id_vars=['exercicio', 'nome', 'sigla', 'class_ranking', 'nota_ranking'],
    value_vars=verificacoes,
    var_name='verificacao',
    value_name='nota'
)

data_verificacoes = data_verificacoes.sort_values(
    by=['exercicio','nome', 'verificacao']
)

df = data_verificacoes.merge(
    df_aplicabilidade,
    on='verificacao',
    how='left'
)

df = df.merge(
    df_descricao,
    on='verificacao',
    how='left'
)

df['dimensao'] = (
    df['verificacao']
    .str[:2]
    .map({
        'D1': 'Dimensão 1',
        'D2': 'Dimensão 2',
        'D3': 'Dimensão 3',
        'D4': 'Dimensão 4'
    })
)

In [52]:
df['aplicavel'] = df.apply(
    lambda row: row[str(row['exercicio'])] != '',
    axis=1
)

# df.head()

In [53]:
df.rename(columns={'nome':'municipio'}, inplace=True)
df.rename(columns={'sigla':'estado'}, inplace=True)

df = df[['exercicio', 'municipio', 'estado', 'verificacao', 'descricao', 'nota', 'aplicavel','class_ranking', 'nota_ranking']]
df.head()

,exercicio,municipio,estado,verificacao,descricao,nota,aplicavel,class_ranking,nota_ranking
0,2019,Abadia de Goiás - GO,GO,D1_00001,Homologação de todos os RREOs,1.0,True,4381,E
1,2019,Abadia de Goiás - GO,GO,D1_00002,Homologação da DCA,1.0,True,4381,E
2,2019,Abadia de Goiás - GO,GO,D1_00003,Homologação de todos os RGFs do poder Executivo,1.0,True,4381,E
3,2019,Abadia de Goiás - GO,GO,D1_00004,Homologação de todos os RGFs do poder Legislativo,1.0,True,4381,E
4,2019,Abadia de Goiás - GO,GO,D1_00006,Tempestividade na homologação dos RREOs,0.0,True,4381,E


In [ ]:
df_ranking = df[
    ['exercicio', 'municipio', 'estado', 'class_ranking', 'nota_ranking']
].copy()

df_ranking = (
    df_ranking
    .drop_duplicates()
    .sort_values(by=['estado', 'municipio', 'exercicio'])
)

df_ranking.to_parquet('dados/dados_tratados/ranking_siconfi.parquet', engine='pyarrow', index=False)

In [56]:
#Visualizar dados exportados

df_ranking

,exercicio,municipio,estado,class_ranking,nota_ranking
5175,2019,Acrelândia - AC,AC,5337,E
1157751,2020,Acrelândia - AC,AC,2941,C
2310327,2021,Acrelândia - AC,AC,1256,B
3462903,2022,Acrelândia - AC,AC,5071,E
4615479,2023,Acrelândia - AC,AC,2192,B
...,...,...,...,...,...
3447171,2021,Xambioá - TO,TO,5406,E
4599747,2022,Xambioá - TO,TO,5350,E
5752323,2023,Xambioá - TO,TO,5217,E
6904899,2024,Xambioá - TO,TO,5091,E
